# 10C0 — Add sccomp + CmdStan to the existing R 4.5.2 environment

An explicit installation notebook, not an automatic upgrade of the DESeq2 setup.
It creates a separate sccomp package-library overlay and writable Stan compilation
cache, using the existing verified R executable. **Bioconductor 3.22** supplies the
compatible sccomp release (2.2.x); the public development docs may describe newer APIs.
The runner checks named function arguments and records actual versions.

CmdStan is a separate compiled dependency, not just an R package. A C++ toolchain,
make, network/repository access and writable disk are required. No `sudo` is used,
no system settings are modified, and existing R libraries are not mass-updated.
Default CmdStan pin is 2.37.0; an already installed compatible CmdStan can be reused.

In [1]:
from pathlib import Path
import sys,json
from IPython.display import display
ADDON_DIR=Path.cwd()  # extracted folder containing tumor_characterization/ and bundled tumor_visibility/
if not (ADDON_DIR/'tumor_characterization').is_dir():raise FileNotFoundError('Set ADDON_DIR to the extracted characterization package.')
sys.path.insert(0,str(ADDON_DIR))
from tumor_characterization import core as tc
if Path(tc.__file__).resolve()!=(ADDON_DIR/'tumor_characterization'/'core.py').resolve():
    raise RuntimeError('Another characterization helper is already imported. Restart the kernel and set ADDON_DIR before continuing.')
pointer=ADDON_DIR/'active_characterization_project.json'
if not pointer.is_file():raise FileNotFoundError('Run 10A first to register the existing 09A project.')
settings=tc.load_json(pointer)
atlas=tc.open_atlas(settings['visibility_project'],settings.get('output_root'),check_sources=True)
print('Source project:',atlas['root'])
print('Characterization output:',atlas['out'])

Source project: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/08_ct26_joint_scvi/CT26_diam50_100_mt10_g50_c100_scvi4k_latent30_seed17/13_tumor_immune_visibility/874ca76b9d58169e
Characterization output: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/08_ct26_joint_scvi/CT26_diam50_100_mt10_g50_c100_scvi4k_latent30_seed17/13_tumor_immune_visibility/874ca76b9d58169e/tumor_characterization/c73c61628af15d1a


In [2]:
BASE_R_CONNECTION=Path('/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_DESeq2_ashr/r_connection.json')
SCCOMP_ENV_ROOT=Path('/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_sccomp')
EXISTING_CMDSTAN_PATH=''    # absolute existing CmdStan directory, or '' to install pinned version
CMDSTAN_VERSION='2.37.0'
INSTALL_SCCOMP=True        # this cell explicitly authorizes downloading/compiling packages
CORES=2
if INSTALL_SCCOMP:
    job=dict(environment_root=str(SCCOMP_ENV_ROOT),cores=CORES,existing_cmdstan_path=EXISTING_CMDSTAN_PATH,cmdstan_version=CMDSTAN_VERSION)
    tc.run_r(ADDON_DIR/'tumor_characterization'/'install_sccomp.R',job,SCCOMP_ENV_ROOT/'setup_logs',BASE_R_CONNECTION)
SCCOMP_CONNECTION=SCCOMP_ENV_ROOT/'sccomp_connection.json'
if not SCCOMP_CONNECTION.is_file():raise FileNotFoundError('Run the installer or supply an existing sccomp_connection.json.')
from tumor_characterization import composition as cc
print('Actual backend:',cc.preflight(SCCOMP_CONNECTION))
tc.save_json(dict(connection_json=str(SCCOMP_CONNECTION)),ADDON_DIR/'active_sccomp_connection.json')

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org
Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31)
Installing package(s) 'sccomp'
also installing the dependency ‘instantiate’

trying URL 'https://cloud.r-project.org/src/contrib/instantiate_0.2.4.tar.gz'
trying URL 'https://bioconductor.org/packages/3.22/bioc/src/contrib/sccomp_2.2.0.tar.gz'
begin installing package ‘instantiate’
* installing *source* package ‘instantiate’ ...
** this is package ‘instantiate’ version ‘0.2.4’
** package ‘instantiate’ successfully unpacked and MD5 sums checked
** using staged installation
** libs
using C compiler: ‘gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0’
make[1]: Entering directory '/tmp/RtmpYUy17m/R.INSTALL2d40836765c33/instantiate/src'
gcc -std=gnu2x -I"/opt/tbio/domino_202511/binaries/R-4.5.2/lib/R/include" -DNDEBUG   -I/opt/tbio/domi

In [5]:
BASE_R_CONNECTION=Path('/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_DESeq2_ashr/r_connection.json')
SCCOMP_ENV_ROOT=Path('/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_sccomp')
EXISTING_CMDSTAN_PATH=''    # absolute existing CmdStan directory, or '' to install pinned version
CMDSTAN_VERSION='2.37.0'
INSTALL_SCCOMP=True        # this cell explicitly authorizes downloading/compiling packages
CORES=2
if INSTALL_SCCOMP:
    job=dict(environment_root=str(SCCOMP_ENV_ROOT),cores=CORES,existing_cmdstan_path=EXISTING_CMDSTAN_PATH,cmdstan_version=CMDSTAN_VERSION)
    tc.run_r(ADDON_DIR/'tumor_characterization'/'install_sccomp.R',job,SCCOMP_ENV_ROOT/'setup_logs',BASE_R_CONNECTION)
SCCOMP_CONNECTION=SCCOMP_ENV_ROOT/'sccomp_connection.json'
if not SCCOMP_CONNECTION.is_file():raise FileNotFoundError('Run the installer or supply an existing sccomp_connection.json.')
from tumor_characterization import composition as cc
print('Actual backend:',cc.preflight(SCCOMP_CONNECTION))
tc.save_json(dict(connection_json=str(SCCOMP_CONNECTION)),ADDON_DIR/'active_sccomp_connection.json')

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org
Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31)
Warning message:
package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'sccomp' 
The C++ toolchain required for CmdStan is setup properly!
CmdStan path set to: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_sccomp/cmdstan/cmdstan-2.37.0
Package/backend installed. Run the synthetic sccomp smoke-test cell next.
Actual backend: {'R': 'R version 4.5.2 (2025-10-31)', 'sccomp': '2.2.0', 'cmdstanr': '0.9.0', 'cmdstan': '2.37.0', 'cmdstan_path': '/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_sccomp/cmdstan/cmdstan-2.37.0', 'estimate_formals': ['.data', 'formula_composition', 'formula_variability', 'sample', 'cell_gro

In [6]:
# Actual package/model test on synthetic counts ONLY. Does not touch CT26 data.
RUN_SCCOMP_SMOKE_TEST=True
if RUN_SCCOMP_SMOKE_TEST:
    smoke=cc.smoke_test(SCCOMP_ENV_ROOT,SCCOMP_CONNECTION,cores=CORES)
    print(json.dumps(smoke,indent=2))
    print('REVIEW_CONVERGENCE means inspect diagnostics; not a failed installation by itself.')

CmdStan path set to: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_sccomp/cmdstan/cmdstan-2.37.0
sccomp says: count column is an integer. The sum-constrained beta binomial model will be used
sccomp says: estimation
sccomp says: the composition design matrix has columns: conditionVehicle, conditionAntiPD1, conditionAdagrasib, conditionCombination
sccomp says: the variability design matrix has columns: (Intercept)
Loading model from cache...
Init values were only set for a subset of parameters. 
Missing init values for the following parameters:
 - chain 1: random_effect_raw, random_effect_raw_2, random_effect_sigma_mu, random_effect_sigma_sigma, random_effect_sigma_raw, sigma_correlation_factor, random_effect_sigma_raw_2, sigma_correlation_factor_2, zero_random_effect
 - chain 2: random_effect_raw, random_effect_raw_2, random_effect_sigma_mu, random_effect_sigma_sigma, random_effect_sigma_raw, sigma_correlation_factor, random_effect_sigma_raw_2, sigma_corr

RuntimeError: R failed (code 1); see /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/env/R452_Bioc322_sccomp/synthetic_sccomp_smoke/556874b8b336/R_run.log. Completed other stages remain intact.

This uses HMC rather than silently relying on a development-version default
approximation. sccomp's version-specific HMC schedule is bounded by the stated
maximum sampling iterations; diagnostics are saved. The smoke test validates the
installed API and compilation path, not full-study convergence or biological validity.

Installation/reference sources:
[sccomp](https://github.com/MangiolaLaboratory/sccomp),
[Bioconductor 3.22 source](https://github.com/bioconductor-source/sccomp/tree/RELEASE_3_22),
[CmdStan installation](https://mc-stan.org/cmdstanr/reference/install_cmdstan.html).